In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor
import torchvision.utils as vutils
import matplotlib.pyplot as plt

In [ ]:
# Define the ContractiveAutoencoder
class ContractiveAutoencoder(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ContractiveAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(output_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, input_size),
            nn.Sigmoid()
        )
        self.beta = 1e-4

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

    def contractive_loss(self, x, encoded):
        # Calculate the Jacobian matrix
        x.requires_grad_(True)
        encoded, _ = self.forward(x)
        jacobian = torch.autograd.functional.jacobian(lambda x: encoded, x)

        # Compute the Frobenius norm of the Jacobian matrix
        jacobian_norm = torch.norm(jacobian, dim=(2, 3))

        # Compute the contractive loss
        contractive_loss = torch.mean(torch.sum(jacobian_norm**2, dim=1))

        return contractive_loss

In [ ]:
def reconstruct_sample(autoencoder, sample_data):
    _, reconstructed_sample = autoencoder(sample_data)
    return reconstructed_sample

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Hyperparameters
input_size = 784
hidden_size = 128
batch_size = 64
num_epochs = 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:


# Load FashionMNIST dataset
dataset = FashionMNIST(root="./data", train=True, transform=ToTensor(), download=True)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Create an instance of the ContractiveAutoencoder
autoencoder = ContractiveAutoencoder(input_size, hidden_size).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.001)

# Training loop
for epoch in range(num_epochs):
    for i, (images, _) in enumerate(data_loader):
        images = images.view(-1, input_size).to(device)

        # Forward pass
        encoded, decoded = autoencoder(images)

        # Calculate the reconstruction loss
        reconstruction_loss = nn.MSELoss()(decoded, images)

        # Calculate the contractive loss
        contractive_loss = autoencoder.contractive_loss(images, encoded)

        # Compute the total loss
        loss = reconstruction_loss + 0.1 * contractive_loss

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 200 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(data_loader)}], Reconstruction Loss: {reconstruction_loss.item():.4f}, Contractive Loss: {contractive_loss.item():.4f}")

print("Training completed!")

# Choose a random sample for reconstruction and visualization
sample_index = 0
sample_data, _ = dataset[sample_index]
sample_data = sample_data.view(1, input_size).to(device)

# Reconstruct the sample
reconstructed_sample = reconstruct_sample(autoencoder, sample_data)

# Visualize the original and reconstructed samples
sample_data = sample_data.cpu().view(1, 28, 28)
reconstructed_sample = reconstructed_sample.cpu().view(1, 28, 28)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title("Original Sample")
plt.imshow(sample_data[0], cmap="gray")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("Reconstructed Sample")
plt.imshow(reconstructed_sample[0], cmap="gray")
plt.axis("off")

plt.tight_layout()
plt.show()
